# Experiment 3: Circuit Discovery & Span Validation

**Criterion 3**: Within-span similarity elevation > population mean + 1σ
**Criterion 4**: Circuit diversity ≥ 60% layer coverage
**Criterion 5**: Class purity distribution is bimodal

Runs the full span-centric circuit discovery pipeline, validates discovered circuits,
and checks multi-circuit membership.

## Colab Setup
Clones the repo, installs dependencies, and mounts Google Drive.

In [ ]:
import os, sys

# 1. Clone repository
REPO_DIR = '/content/trainable-circuits'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/jacobposchl/trainable-circuits {REPO_DIR}

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# 2. Install extra dependencies
!pip install hdbscan umap-learn -q

# 3. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 4. Path constants
DRIVE_DIR  = '/content/drive/MyDrive/ctls'
DATA_DIR   = DRIVE_DIR + '/data/raw'
CKPT_DIR   = DRIVE_DIR + '/experiments'
CONFIG_DIR = REPO_DIR  + '/configs'

print('Repo:     ', REPO_DIR)
print('Data:     ', DATA_DIR)
print('Checkpts: ', CKPT_DIR)

In [ ]:
import torch
import yaml
import numpy as np
import matplotlib.pyplot as plt
from models.backbone import FrozenBackbone
from models.meta_encoder import MetaEncoder
from data.cifar import get_standard_loaders
from evaluation.circuit_analysis import CircuitAnalyzer
from evaluation.discovery import SpanCentricDiscovery
from evaluation.metrics import (
    within_span_elevation,
    circuit_diversity,
    class_purity_distribution,
)
from evaluation.circuit_viz import (
    plot_span_coverage,
    plot_multi_circuit_histogram,
    plot_circuit_members,
)
from evaluation.circuit_analysis import denormalize

In [ ]:
# Load config and override data directory
config_path     = CONFIG_DIR + '/phase1.yaml'
checkpoint_path = CKPT_DIR  + '/phase1/best.pt'

with open(config_path) as f:
    config = yaml.safe_load(f)

config['data']['data_dir'] = DATA_DIR

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
# Build models and load weights
flow_cfg = config['model'].get('flow_compression', {})

backbone = FrozenBackbone(
    arch=config['model']['arch'],
    num_classes=config['model']['num_classes'],
    pretrained=config['model']['pretrained'],
    grid_size=flow_cfg.get('grid_size', 4),
    flow_dim=flow_cfg.get('flow_dim', 256),
).to(device)

meta_encoder = MetaEncoder(
    layer_dims=backbone.layer_dims,
    projection_dim=config['model']['meta_encoder']['projection_dim'],
    n_heads=config['model']['meta_encoder']['n_heads'],
    n_transformer_layers=config['model']['meta_encoder']['n_transformer_layers'],
    dropout=config['model']['meta_encoder']['dropout']
).to(device)

ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
meta_encoder.load_state_dict(ckpt['meta_encoder_state'])
meta_encoder.eval()
print('Model loaded.')

In [ ]:
# Create validation loader and collect representations
_, val_loader = get_standard_loaders(
    data_dir=DATA_DIR,
    batch_size=config['data'].get('batch_size', 256),
    num_workers=0,
    augment=False,
    download=False,
)

analyzer = CircuitAnalyzer(backbone, meta_encoder, val_loader, device)
data     = analyzer.collect_representations(max_samples=2000)
N = data['labels'].shape[0]
L = len(data['z_list'])
print(f'Collected {N} samples, {L} layers')

# Convert z_list to numpy — one [N, d] array per layer
z_list_np  = [z.numpy() for z in data['z_list']]
labels_np  = data['labels'].numpy()

In [ ]:
# Run image-centric circuit discovery: UMAP + HDBSCAN per span
disc_cfg  = config['discovery']
discovery = SpanCentricDiscovery(
    n_layers=L,
    umap_n_components=disc_cfg.get('umap_n_components', 15),
    umap_n_neighbors=disc_cfg.get('umap_n_neighbors', 15),
    min_cluster_fraction=disc_cfg['min_cluster_fraction'],
    max_cluster_fraction=disc_cfg.get('max_cluster_fraction', 0.40),
    min_cluster_size=disc_cfg['hdbscan_min_cluster_size'],
)

circuits = discovery.discover_all(z_list_np)
print(f'Discovered {len(circuits)} canonical circuits')

In [ ]:
# Criterion 3: Within-span similarity elevation
# Computes elevation + purity for every circuit and stores in circuit_stats.
# ── parameters ──────────────────────────────────────────────────────────────
ELEVATION_THRESHOLD = 1.0   # σ — circuits at or above this pass C3
# ────────────────────────────────────────────────────────────────────────────

CIFAR10_CLASSES = ['airplane','automobile','bird','cat','deer',
                   'dog','frog','horse','ship','truck']

print("Computing elevation and purity for all circuits (may take a minute)...")
circuit_stats = []
for i, circuit in enumerate(circuits):
    pop_sims     = discovery.compute_span_similarities(z_list_np, circuit['span'])
    cluster_sims = discovery.compute_span_similarities(z_list_np, circuit['span'],
                                                        image_mask=circuit['image_mask'])
    elev   = within_span_elevation(cluster_sims, pop_sims)['elevation_sigma']
    purity = SpanCentricDiscovery.compute_class_purity(circuit, labels_np)

    circuit_labels = labels_np[circuit['image_mask']]
    counts = np.bincount(circuit_labels, minlength=len(CIFAR10_CLASSES))
    dominant_class = CIFAR10_CLASSES[counts.argmax()]
    l_start, l_end = circuit['span']

    circuit_stats.append({
        'idx':      i,
        'span':     circuit['span'],
        'span_len': l_end - l_start + 1,
        'size':     circuit['size'],
        'elevation': elev,
        'passes_c3': elev >= ELEVATION_THRESHOLD,
        'purity':   purity,
        'dominant': dominant_class,
    })

n_pass = sum(s['passes_c3'] for s in circuit_stats)
elevs  = [s['elevation'] for s in circuit_stats]
print(f"Total circuits   : {len(circuits)}")
print(f"C3 pass (≥{ELEVATION_THRESHOLD}σ)  : {n_pass} / {len(circuits)}")
print(f"Elevation range  : {min(elevs):.2f}σ – {max(elevs):.2f}σ  "
      f"(mean {np.mean(elevs):.2f}σ)")

In [ ]:
# Criterion 4: Circuit diversity
# circuit_diversity expects list of (l_start, l_end) spans
spans  = [c['span'] for c in circuits]
result = circuit_diversity(spans, L)
print(f"Layer coverage: {result['coverage']:.1%}  "
      f"(target \u2265 60%, {'PASS' if result['passes'] else 'FAIL'})")

fig = plot_span_coverage(circuits, L)
plt.tight_layout()
plt.show()

In [ ]:
# Criterion 5: Class purity distribution
purities = [
    SpanCentricDiscovery.compute_class_purity(c, labels_np)
    for c in circuits
]

result = class_purity_distribution(purities)
print(f"Agnostic (<0.3): {result['n_agnostic']}, "
      f"Specific (>0.7): {result['n_specific']}  "
      f"({'PASS' if result['passes'] else 'FAIL'})")

In [ ]:
# Multi-circuit membership distribution (per image)
counts = SpanCentricDiscovery.multi_circuit_membership(circuits, n_images=N)

fig = plot_multi_circuit_histogram(counts)
plt.tight_layout()
plt.show()

In [ ]:
# Visualisation helper + top circuits by a chosen metric.
# ── parameters ──────────────────────────────────────────────────────────────
N_TOP      = 3          # number of circuits to show
MAX_IMAGES = 16         # images per circuit grid
SORT_BY    = 'elevation'  # 'elevation' | 'size' | 'purity' | 'span_len'
# ────────────────────────────────────────────────────────────────────────────

def show_circuits(stats_subset, max_per=3, max_images=16, title=""):
    """
    Render a grid of member images for each circuit in stats_subset.

    Each image caption shows its class label and per-layer mean z-similarity
    to the rest of the cluster (the 'activation profile').

    Args:
        stats_subset : list of circuit_stats dicts (each must have 'idx')
        max_per      : maximum circuits to render
        max_images   : maximum images shown per circuit
        title        : optional section header
    """
    if title:
        print(f"\n{'='*60}\n{title}\n{'='*60}")
    if not stats_subset:
        print("  (no circuits match the filter)")
        return
    for stat in stats_subset[:max_per]:
        circuit  = circuits[stat['idx']]
        show_idx = np.where(circuit['image_mask'])[0][:max_images]
        n_show   = len(show_idx)

        # per-image mean z-similarity profile: how similar is each image
        # to the cluster centroid at each layer?
        input_profiles = np.zeros((n_show, L))
        for l in range(L):
            z_show    = z_list_np[l][show_idx]
            z_cluster = z_list_np[l][circuit['image_mask']]
            input_profiles[:, l] = (z_show @ z_cluster.T).mean(axis=1)

        imgs_np = denormalize(data['images'][show_idx]).numpy()
        lbls_np = labels_np[show_idx]

        fig = plot_circuit_members(imgs_np, lbls_np, input_profiles, span=circuit['span'])
        plt.suptitle(
            f"Circuit {stat['idx']}  span={stat['span']}  n={stat['size']}  "
            f"elev={stat['elevation']:.2f}σ  purity={stat['purity']:.2f}  "
            f"dominant={stat['dominant']}",
            y=1.01,
        )
        plt.tight_layout()
        plt.show()


# Show top N circuits sorted by chosen metric
top = sorted(circuit_stats, key=lambda s: -s[SORT_BY])
show_circuits(top, max_per=N_TOP, max_images=MAX_IMAGES,
              title=f"Top {N_TOP} circuits by {SORT_BY}")

In [ ]:
# High-quality circuits: strongest cluster signal above elevation threshold.
# Lower MIN_ELEVATION to see more; raise it to see only the sharpest circuits.
# ── adjust ──────────────────────────────────────────────────────────────────
MIN_ELEVATION = 1.5   # σ — quality floor
SORT_BY       = 'elevation'   # 'elevation' | 'purity' | 'size' | 'span_len'
N_SHOW        = 4
MAX_IMAGES    = 16
# ────────────────────────────────────────────────────────────────────────────

filtered = [s for s in circuit_stats if s['elevation'] >= MIN_ELEVATION]
filtered.sort(key=lambda s: -s[SORT_BY])

print(f"High-quality circuits (elev ≥ {MIN_ELEVATION}σ): {len(filtered)} total")
for s in filtered[:10]:
    print(f"  Circuit {s['idx']} span={s['span']} elev={s['elevation']:.2f}σ "
          f"purity={s['purity']:.2f} dominant={s['dominant']} n={s['size']}")

show_circuits(filtered, max_per=N_SHOW, max_images=MAX_IMAGES,
              title=f"High-Quality Circuits (elevation ≥ {MIN_ELEVATION}σ)")

In [ ]:
# Long-span circuits: these activate across many consecutive layers,
# suggesting a persistent or slowly-evolving computation in the backbone.
# ── adjust ──────────────────────────────────────────────────────────────────
MIN_SPAN_LEN  = 5     # minimum number of layers spanned
MIN_ELEVATION = 0.0   # σ — set to e.g. 1.0 to add quality filter
N_SHOW        = 4
MAX_IMAGES    = 16
# ────────────────────────────────────────────────────────────────────────────

filtered = [s for s in circuit_stats
            if s['span_len'] >= MIN_SPAN_LEN and s['elevation'] >= MIN_ELEVATION]
filtered.sort(key=lambda s: (-s['span_len'], -s['elevation']))

print(f"Long-span circuits (≥ {MIN_SPAN_LEN} layers): {len(filtered)} total")
for s in filtered[:12]:
    print(f"  Circuit {s['idx']} span={s['span']} len={s['span_len']} "
          f"elev={s['elevation']:.2f}σ purity={s['purity']:.2f} n={s['size']}")

show_circuits(filtered, max_per=N_SHOW, max_images=MAX_IMAGES,
              title=f"Long-Span Circuits (≥ {MIN_SPAN_LEN} layers)")

In [ ]:
# Class-specific circuits: most of the cluster images share one class label.
# Set CLASS_FILTER to focus on a particular class (e.g. 'bird'), or None for all.
# ── adjust ──────────────────────────────────────────────────────────────────
MIN_PURITY   = 0.7          # fraction of images from one class
MIN_ELEVATION = 0.0         # σ — set to e.g. 1.0 to require quality too
CLASS_FILTER = None         # e.g. 'bird' | 'automobile' | None = all classes
N_SHOW       = 4
MAX_IMAGES   = 16
# ────────────────────────────────────────────────────────────────────────────

filtered = [s for s in circuit_stats
            if s['purity'] >= MIN_PURITY
            and s['elevation'] >= MIN_ELEVATION
            and (CLASS_FILTER is None or s['dominant'] == CLASS_FILTER)]
filtered.sort(key=lambda s: -s['purity'])

print(f"Class-specific circuits (purity ≥ {MIN_PURITY}): {len(filtered)} total")
for s in filtered:
    print(f"  Circuit {s['idx']} span={s['span']} purity={s['purity']:.2f} "
          f"dominant={s['dominant']} elev={s['elevation']:.2f}σ n={s['size']}")

show_circuits(filtered, max_per=N_SHOW, max_images=MAX_IMAGES,
              title=f"Class-Specific Circuits (purity ≥ {MIN_PURITY}"
                    + (f", class={CLASS_FILTER}" if CLASS_FILTER else "") + ")")

In [ ]:
# Cross-class agnostic circuits: low purity = visual concept shared across classes.
# These are the most interesting structurally — what does the network compute
# that isn't tied to a single category?
# ── adjust ──────────────────────────────────────────────────────────────────
MAX_PURITY    = 0.3   # must have ≤ this fraction from dominant class
MIN_ELEVATION = 1.0   # σ — only show circuits with genuine cluster signal
N_SHOW        = 4
MAX_IMAGES    = 16
# ────────────────────────────────────────────────────────────────────────────

filtered = [s for s in circuit_stats
            if s['purity'] <= MAX_PURITY and s['elevation'] >= MIN_ELEVATION]
filtered.sort(key=lambda s: (-s['elevation'], s['purity']))

print(f"Cross-class agnostic circuits "
      f"(purity ≤ {MAX_PURITY}, elev ≥ {MIN_ELEVATION}σ): {len(filtered)} total")
for s in filtered[:10]:
    print(f"  Circuit {s['idx']} span={s['span']} "
          f"elev={s['elevation']:.2f}σ purity={s['purity']:.2f} "
          f"dominant={s['dominant']} n={s['size']}")

show_circuits(filtered, max_per=N_SHOW, max_images=MAX_IMAGES,
              title=f"Cross-Class Agnostic Circuits "
                    f"(purity ≤ {MAX_PURITY}, elev ≥ {MIN_ELEVATION}σ)")

In [ ]:
# Span heatmap: shows where circuits are found and how good they are.
# Each cell (l_start, l_end) is coloured by the mean value of METRIC across
# all circuits in that span.  The number in each cell is the circuit count.
# ── adjust ──────────────────────────────────────────────────────────────────
METRIC = 'elevation'   # 'elevation' | 'purity' | 'size' | 'span_len'
CMAP   = 'viridis'     # any matplotlib colormap name
# ────────────────────────────────────────────────────────────────────────────

sum_grid   = np.zeros((L, L))
count_grid = np.zeros((L, L), dtype=int)
for s in circuit_stats:
    r, c = s['span']
    sum_grid[r, c]   += s[METRIC]
    count_grid[r, c] += 1

grid = np.where(count_grid > 0, sum_grid / count_grid, np.nan)

cmap_obj = plt.cm.get_cmap(CMAP).copy()
cmap_obj.set_bad(color='#e0e0e0')   # grey = no circuits at this span

fig, ax = plt.subplots(figsize=(max(6, L + 1), max(5, L)))
im = ax.imshow(grid, origin='upper', aspect='auto', cmap=cmap_obj,
               vmin=np.nanmin(grid), vmax=np.nanmax(grid))
plt.colorbar(im, ax=ax, label=f'mean {METRIC}')
ax.set_xlabel('l_end')
ax.set_ylabel('l_start')
ax.set_xticks(range(L));  ax.set_xticklabels(range(L))
ax.set_yticks(range(L));  ax.set_yticklabels(range(L))
ax.set_title(f'Circuit density & mean {METRIC} by span\n'
             f'(cell value = circuit count)')

for r in range(L):
    for c in range(r, L):
        n = count_grid[r, c]
        if n > 0:
            ax.text(c, r, str(n), ha='center', va='center',
                    fontsize=8, color='white', fontweight='bold')

plt.tight_layout()
plt.show()